# GBM & Itô Correction

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Geometric Brownian motion paths vs e^{μt} and the Itô drift correction.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/gbm-ito/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: dX = mu X dt + sigma X dW, X0 = 1
T, dt, mu, sigma, x0 = 2.0, 0.01, 0.3, 0.5, 1.0
n_paths, steps = 100000, round(T / dt)

# ── Exact solution: X_t = X0 exp((mu - sigma^2/2) t + sigma W_t) ────────────
dW = math.sqrt(dt) * torch.randn(n_paths, steps, dtype=torch.float64)
W = dW.sum(1)
X_exact = x0 * torch.exp((mu - 0.5 * sigma ** 2) * T + sigma * W)

# Euler-Maruyama on the same noise for comparison
X = torch.full((n_paths,), x0, dtype=torch.float64)
for k in range(steps):
    X = X + mu * X * dt + sigma * X * dW[:, k]

print(f"E[X_T]: exact solution {X_exact.mean().item():.4f}, Euler-Maruyama {X.mean().item():.4f}, "
      f"theory x0 e^(mu T) = {x0 * math.exp(mu * T):.4f}")
print(f"median: {X_exact.median().item():.4f}  theory x0 e^((mu - sigma^2/2) T) = {x0 * math.exp((mu - 0.5 * sigma ** 2) * T):.4f}")
print(f"P(X_T < X0) = {(X_exact < x0).double().mean().item():.4f} even though the mean grows")
print(f"mean |EM - exact| per path = {(X - X_exact).abs().mean().item():.4f}")

# ── log X_T is Gaussian: mean (mu - sigma^2/2) T, variance sigma^2 T ────────
logX = X_exact.log()
print(f"\nlog X_T: mean {logX.mean().item():.4f} ({(mu - 0.5 * sigma ** 2) * T:.4f}), variance {logX.var().item():.4f} ({sigma ** 2 * T:.4f})")

E[X_T]: exact solution 1.8202, Euler-Maruyama 1.8188, theory x0 e^(mu T) = 1.8221
median: 1.4180  theory x0 e^((mu - sigma^2/2) T) = 1.4191
P(X_T < X0) = 0.3101 even though the mean grows
mean |EM - exact| per path = 0.0362

log X_T: mean 0.3494 (0.3500), variance 0.5005 (0.5000)
